# GPU checkpoint and restore, with a real model

Pausing a Workspace checkpoints the pod; resuming restores it. This notebook checks that the
checkpoint includes **live GPU state** — that a multi-gigabyte model sitting in VRAM comes back
intact, in the same process, without being reloaded.

That is the whole point of the feature for AI work. Loading a model is the slow part of starting a
notebook; if the checkpoint captures it, resuming skips the load entirely.

We use **Qwen2.5-3B-Instruct** in fp16 (~6 GB of VRAM). The test is deliberately hard to fake:

- **Weights must be bit-identical.** A hash of a weight tensor read straight out of VRAM has to
  match exactly. This is static data, so there is no floating-point wiggle room to hide behind.
- **The process must be the same one.** A matching pid rules out a restart.
- **Generation must be reproducible.** Greedy decoding on the same prompt has to produce the same
  token ids.
- **The GPU must still be usable.** A brand-new prompt has to work after the restore, proving the
  CUDA context rebound rather than merely surviving as bytes.

**Requirements:** the **GPU** image and the **GPU T4 Spot** pod config, on a snapshot-enabled
`WorkspaceKind`. T4 is Turing, so this notebook uses fp16 — bf16 is not supported on that card.

**How to run:** run steps 1–2, pause the Workspace, resume it, then run **only** step 3. Re-running
steps 1–2 after the resume would reload the model and defeat the test.

In [ ]:
# Step 1 — confirm we have a GPU with room for the model.
import os
import socket
import time

import torch

print(f"host       : {socket.gethostname()}")
print(f"pid        : {os.getpid()}")
print(f"torch      : {torch.__version__}")
print(f"cuda build : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device is visible, so there is nothing to checkpoint. Recreate this "
        "Workspace with the GPU image and the 'GPU T4 Spot' pod config."
    )

DEVICE = torch.device("cuda:0")
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"device     : {torch.cuda.get_device_name(0)} ({vram:.0f} GiB VRAM)")
print(f"capability : {torch.cuda.get_device_capability(0)}")

# Weights land on the home PVC, so they survive pod restarts and only download once.
os.environ.setdefault("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
print(f"hf cache   : {os.environ['HF_HOME']}")

In [ ]:
# Step 2 — load the model onto the GPU and record exactly what is there.
import hashlib
import threading

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PROMPT = "In one sentence, what is a GPU good at?"

PID = os.getpid()
CREATED_AT = time.time()

# The first run downloads ~6 GB into the home PVC; later runs read it from there.
started = time.time()
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID)
MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # T4 is Turing: fp16 yes, bf16 no. On transformers 4.x this argument was
    # spelled torch_dtype; the GPU image pins transformers 5.x, which renamed it.
    dtype=torch.float16,
).to(DEVICE)
MODEL.eval()
LOAD_SECONDS = time.time() - started


def weight_fingerprint(model):
    """Bit-exact hash of a weight tensor read back out of VRAM.

    Weights are static, so unlike anything computed this has no floating-point
    tolerance to hide behind: it either comes back identical or it does not.
    """
    tensor = model.model.layers[0].self_attn.q_proj.weight
    return hashlib.sha256(tensor.detach().cpu().numpy().tobytes()).hexdigest()[:16]


def greedy(prompt, max_new_tokens=40):
    """Deterministic generation, so the same prompt must give the same token ids."""
    chat = TOKENIZER.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = TOKENIZER(chat, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return generated.tolist(), TOKENIZER.decode(generated, skip_special_tokens=True)


WEIGHT_FP = weight_fingerprint(MODEL)
started = time.time()
OUTPUT_IDS, OUTPUT_TEXT = greedy(PROMPT)
GENERATE_SECONDS = time.time() - started

# Keeps counting while the process runs, so after the resume we can tell how long it
# was actually frozen.
TICKS = 0


def _tick():
    global TICKS
    while True:
        time.sleep(1)
        TICKS += 1


threading.Thread(target=_tick, daemon=True).start()

print(f"model       : {MODEL_ID}")
print(f"parameters  : {sum(p.numel() for p in MODEL.parameters()) / 1e9:.2f} B")
print(f"load time   : {LOAD_SECONDS:.1f}s   <- this is what a resume should save you")
print(f"vram in use : {torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated, "
      f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB reserved")
print(f"weight hash : {WEIGHT_FP}")
print(f"pid         : {PID}")
print()
print(f"prompt      : {PROMPT}")
print(f"generated   : {OUTPUT_TEXT.strip()}   ({GENERATE_SECONDS:.1f}s)")

## Now pause the Workspace

The model is now resident in VRAM. Note the `weight hash` printed above, then pause — either with
the **Pause** button in the Workspaces UI, or from a terminal with access to the cluster:

```bash
kubectl -n <namespace> patch workspace <name> --type=merge -p '{"spec":{"paused":true}}'
```

The pause is not instant: the addon holds `spec.paused=false` until GKE has finished writing the
snapshot, and a pod holding several GB of VRAM takes longer than an idle one. Watch it settle:

```bash
kubectl -n <namespace> get workspace <name> \
  -o jsonpath='{.spec.paused}{" "}{.metadata.annotations.podsnapshot\.gke\.kubeflow\.org/checkpoint-state}{"\n"}'
# -> "true Ready" once the checkpoint is safely in GCS
```

Then resume it (**Resume** in the UI, or set `spec.paused` back to `false`), wait for the pod to be
`1/1`, reopen this notebook, and run step 3 below **without re-running steps 1 and 2**.

In [ ]:
# Step 3 — run this AFTER the resume. Do not re-run the cells above.
try:
    MODEL, TOKENIZER, PID, WEIGHT_FP, OUTPUT_IDS, LOAD_SECONDS
except NameError:
    raise RuntimeError(
        "The model and globals from step 2 are gone, so this is a fresh kernel: the pod "
        "was restarted rather than restored. GPU checkpoint/restore FAILED."
    ) from None

checks = []

current_pid = os.getpid()
checks.append(("same process", current_pid == PID, f"pid {current_pid}, was {PID}"))

# The decisive one: model weights read back out of VRAM, bit for bit.
restored_fp = weight_fingerprint(MODEL)
checks.append(
    ("weights bit-identical", restored_fp == WEIGHT_FP, f"{restored_fp}, was {WEIGHT_FP}")
)

# Same prompt, greedy decoding -> the model must say exactly the same thing.
started = time.time()
replay_ids, replay_text = greedy(PROMPT)
replay_seconds = time.time() - started
checks.append(
    (
        "generation reproducible",
        replay_ids == OUTPUT_IDS,
        f"{len(replay_ids)} tokens, identical" if replay_ids == OUTPUT_IDS else "output diverged",
    )
)

# A prompt it has never seen, to prove the CUDA context rebound rather than just
# surviving as bytes.
try:
    _, fresh_text = greedy("Name one city in Japan.", max_new_tokens=16)
    usable, detail = True, f"new prompt -> {fresh_text.strip()[:60]!r}"
except Exception as exc:  # noqa: BLE001 - report whatever the failure mode is
    usable, detail = False, f"{type(exc).__name__}: {exc}"
checks.append(("cuda context usable", usable, detail))

# Wall clock that passed without the ticker advancing is time spent checkpointed.
frozen = time.time() - CREATED_AT - TICKS
checks.append(("process was frozen", frozen > 5, f"~{frozen:.0f}s paused, {TICKS}s running"))

label_width = max(len(name) for name, _, _ in checks)
for name, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name.ljust(label_width)}   {detail}")

passed = all(ok for _, ok, _ in checks)
print()
print(f"GPU checkpoint/restore: {'PASS' if passed else 'FAIL'}")
if passed:
    print(
        f"\nThe model was already resident: first generation after the resume took "
        f"{replay_seconds:.1f}s, against {LOAD_SECONDS:.1f}s to load it from scratch."
    )

## Reading the result

| Failure | What it means |
|---|---|
| `NameError` / fresh kernel | The pod was recreated, not restored. Either the Workspace is not snapshot-enabled, or the restore did not happen. |
| `same process` fails | A new process took over. Same conclusion as above. |
| `weights bit-identical` fails | The process survived but VRAM did not come back intact — a genuine GPU checkpoint bug, and the most serious result here. |
| `generation reproducible` fails | Weights match but compute does not. Suspect the CUDA context, cuBLAS workspaces, or RNG state. |
| `cuda context usable` fails | The bytes survived but the context did not rebind to the device. |
| `process was frozen` fails | Nothing was actually paused; check that the pause completed before you resumed. |

### Measured on a T4 (n1-standard-16, Qwen2.5-3B-Instruct in fp16)

| | |
|---|---|
| Model load from scratch | 22–38 s |
| VRAM resident | 5.86 GiB |
| **Checkpoint (pause)** | **115–127 s**, 8.3 GiB written to GCS |
| **Restore (resume)** | **11–12 s** |
| First generation after resume | 2.0 s |

The asymmetry is the point: a pause costs roughly two minutes, but a resume costs twelve seconds and
hands back a model that would otherwise take half a minute to reload. Note that the snapshot in GCS
(8.3 GiB) is larger than the VRAM in use (5.86 GiB) — it holds process memory as well.

To watch the machinery from a terminal with cluster access:

```bash
# The snapshot taken for this Workspace, and where it went.
kubectl -n <namespace> get podsnapshots

# What the addon decided, live.
kubectl -n kubeflow-workspaces logs -l app=gke-workspace-snapshot-addon -f --prefix
```

### Known constraints

> **Only the container's own processes are checkpointed.** A process started with `kubectl exec`
> is not part of the checkpointed process tree and is killed by the pause; the JupyterLab server and
> the kernels it owns survive. So run this notebook in JupyterLab. Verifying from an `exec` shell
> will show a dead process and look like a checkpoint failure when nothing is wrong.

> **Memory during the pause.** GPU state is copied through the pod's process memory while
> checkpointing, so peak pod memory during a pause is well above what the workload uses at rest.
> The `GPU T4 Spot` pod config asks for 40 GB on an n1-standard-16 partly for that reason.

> **The pod never leaves `Pending`.** Snapshot-enabled pods are mutated to run under
> `runtimeClassName=gvisor`, so the node must offer both a T4 and the gVisor sandbox. GKE Sandbox
> does support NVIDIA GPUs (via `nvproxy`, GKE 1.29.2+) and node auto-provisioning will build such a
> node, but the first one takes several minutes. `describe pod` shows the unsatisfied selector.

> **fp16, not bf16.** T4 is Turing (sm_75). Use fp16 on this card.

Other limits worth knowing: multi-GPU pods are supported only on L4, MIG-shared GPUs are not
supported at all, and a snapshot cannot be restored onto a different GPU type.